<a href="https://colab.research.google.com/github/fromofrancishaba/2024_Cours_Projet_Statistique_Sous_R_ENSAE/blob/main/reforme_terminer_inge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 1. INSTALLATION DES DÉPENDANCES ET CONFIGURATION DU TUNNEL
# ==============================================================================
!pip install -q streamlit pandas openpyxl plotly scikit-learn

# Création d'un tunnel stable avec localtunnel (sans mot de passe requis pour les tiers)
import os
import subprocess
import time

# Récupération de l'adresse IP publique (au cas où elle serait demandée par sécurité)
import urllib
print("----------------------------------------------------------------")
print("SI LE TUNNEL DEMANDE UN MOT DE PASSE (PASSWORD) À VOS UTILISATEURS,")
print("Saisissez cette IP :", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
print("----------------------------------------------------------------")

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
import numpy as np
from io import BytesIO

# Configuration de la page pour un rendu professionnel
st.set_page_config(page_title="Gestion de Tontine Pro", page_icon="💰", layout="wide")

st.title("💰 Application de Gestion et Prédiction de Tontine")
st.markdown("Optimisez la gestion des cotisations, visualisez les flux et anticipez les collectes.")

# --- FONCTION UTILITAIRE : GÉNÉRATION DEMO ---
def load_demo_data():
    data = {
        "Date": pd.date_range(start="2026-01-01", periods=12, freq="ME"),
        "Membre": ["Amadou", "Fatou", "Moussa", "Mariama"] * 3,
        "Montant_Cotise": [50000, 60000, 45000, 55000, 52000, 61000, 47000, 56000, 55000, 63000, 48000, 58000],
        "Statut": ["Payé", "Payé", "En retard", "Payé"] * 3
    }
    return pd.DataFrame(data)

# --- BARRE LATÉRALE : IMPORTATION MULTI-FORMATS ---
st.sidebar.header("📥 1. Importation des Données")
uploaded_file = st.sidebar.file_uploader("Téléversez votre fichier Excel (.xlsx, .xls) ou CSV", type=["xlsx", "xls", "csv"])

# Chargement sécurisé du dataframe
if uploaded_file is not None:
    try:
        if uploaded_file.name.endswith('.csv'):
            df_raw = pd.read_csv(uploaded_file)
        else:
            df_raw = pd.read_excel(uploaded_file, engine='openpyxl')
        st.sidebar.success("✅ Fichier importé avec succès !")
    except Exception as e:
        st.sidebar.error(f"❌ Erreur de lecture : {e}")
        df_raw = load_demo_data()
else:
    st.sidebar.info("💡 Utilisation des données de démonstration.")
    df_raw = load_demo_data()

# --- NETTOYAGE ET CORRESPONDANCE DES COLONNES ---
st.sidebar.subheader("⚙️ Configuration des colonnes")
df_raw.columns = [str(c).strip() for c in df_raw.columns]
cols = list(df_raw.columns)

def auto_detect(options, keywords):
    for kw in keywords:
        for i, opt in enumerate(options):
            if kw.lower() in opt.lower():
                return i
    return 0

col_date = st.sidebar.selectbox("Colonne 'Date'", cols, index=auto_detect(cols, ["date", "période"]))
col_membre = st.sidebar.selectbox("Colonne 'Membre'", cols, index=auto_detect(cols, ["membre", "nom", "adhérent"]))
col_montant = st.sidebar.selectbox("Colonne 'Montant'", cols, index=auto_detect(cols, ["montant", "cotise", "versement", "argent"]))
col_statut = st.sidebar.selectbox("Colonne 'Statut'", cols, index=auto_detect(cols, ["statut", "état", "situation"]))

# Reconstruction du DataFrame standardisé
df = df_raw.rename(columns={
    col_date: 'Date',
    col_membre: 'Membre',
    col_montant: 'Montant_Cotise',
    col_statut: 'Statut'
})

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])
df['Montant_Cotise'] = pd.to_numeric(df['Montant_Cotise'], errors='coerce').fillna(0)

# --- FILTRES DYNAMIQUES ---
st.sidebar.header("🎛️ 2. Filtres d'affichage")
membres_dispos = ["Tous"] + list(df['Membre'].unique())
choix_membre = st.sidebar.selectbox("Filtrer par Membre", membres_dispos)

statuts_dispos = ["Tous"] + list(df['Statut'].unique())
choix_statut = st.sidebar.selectbox("Filtrer par Statut", statuts_dispos)

df_filtered = df.copy()
if choix_membre != "Tous":
    df_filtered = df_filtered[df_filtered['Membre'] == choix_membre]
if choix_statut != "Tous":
    df_filtered = df_filtered[df_filtered['Statut'] == choix_statut]

# --- SECTION 1 : DASHBOARD KPI ---
st.header("📊 Tableau de Bord Financier")
col1, col2, col3 = st.columns(3)

total_collecte = df_filtered['Montant_Cotise'].sum()
nb_membres = df_filtered['Membre'].nunique()

with col1:
    st.metric("Total des Dépôts Filtrés", f"{total_collecte:,.0f} FCFA")
with col2:
    st.metric("Membres Sélectionnés", nb_membres)
with col3:
    st.metric("Dernière Opération", str(df_filtered['Date'].max().strftime('%d/%m/%Y')) if not df_filtered.empty else "N/A")

st.markdown("---")

# --- SECTION 2 : VISUALISATIONS HAUTE PERFORMANCE ---
st.header("📈 Analyses Graphiques")
g1, g2 = st.columns(2)

with g1:
    st.subheader("Répartition des Cotisations")
    if not df_filtered.empty:
        fig_bar = px.bar(df_filtered, x='Membre', y='Montant_Cotise', color='Statut',
                         title="Somme des versements par membre", barmode='group', template="plotly_white")
        st.plotly_chart(fig_bar, use_container_width=True)
    else:
        st.warning("Aucune donnée disponible.")

with g2:
    st.subheader("Évolution Temporelle des Flux")
    if not df_filtered.empty:
        df_chrono = df_filtered.groupby('Date')['Montant_Cotise'].sum().reset_index()
        fig_line = px.line(df_chrono, x='Date', y='Montant_Cotise', markers=True,
                           title="Chronologie globale des dépôts", template="plotly_white")
        st.plotly_chart(fig_line, use_container_width=True)
    else:
        st.warning("Aucune donnée chronologique disponible.")

st.markdown("---")

# --- SECTION 3 : PRÉDICTION PRÉCISE VIA ML ---
st.header("🔮 Prédiction des Prochaines Collectes (Tendance Algorithmique)")

df_pred = df.groupby('Date')['Montant_Cotise'].sum().reset_index()
df_futur = pd.DataFrame()

if len(df_pred) > 1:
    df_pred['Timestamp'] = df_pred['Date'].astype(np.int64) // 10**9
    X = df_pred[['Timestamp']]
    y = df_pred['Montant_Cotise']

    model = LinearRegression()
    model.fit(X, y)

    derniere_date = df_pred['Date'].max()
    futures_dates = pd.date_range(start=derniere_date, periods=4, freq="ME")[1:]
    futures_timestamps = futures_dates.astype(np.int64) // 10**9

    predictions = np.clip(model.predict(futures_timestamps.values.reshape(-1, 1)), 0, None)
    df_futur = pd.DataFrame({'Date': futures_dates, 'Montant_Estime_FCFA': predictions})

    fig_pred = go.Figure()
    fig_pred.add_trace(go.Scatter(x=df_pred['Date'], y=df_pred['Montant_Cotise'], name="Historique Réel", mode="lines+markers"))
    fig_pred.add_trace(go.Scatter(x=df_futur['Date'], y=df_futur['Montant_Estime_FCFA'], name="Prédiction IA", mode="lines+markers", line=dict(dash='dash', color='red')))
    fig_pred.update_layout(title="Modèle prédictif linéaire des collectes", template="plotly_white")
    st.plotly_chart(fig_pred, use_container_width=True)

    st.write("📈 **Montants prévisionnels calculés :**")
    for d, p in zip(futures_dates, predictions):
        st.write(f"- Échéance du **{d.strftime('%d/%m/%Y')}** : environ **{int(p):,} FCFA**")
else:
    st.info("Historique temporel insuffisant pour projeter des tendances de prédiction.")

st.markdown("---")

# --- SECTION 4 : MODULE D'EXPORTATION MULTI-ONGLETS ---
st.header("💾 Centre de Téléchargement des Résultats")

def generate_global_report(df_history, df_predictions):
    output = BytesIO()
    with pd.ExcelWriter(output, engine='openpyxl') as writer:
        df_history.to_excel(writer, index=False, sheet_name='Historique de Tontine')
        if not df_predictions.empty:
            df_export_pred = df_predictions.copy()
            df_export_pred['Date'] = df_export_pred['Date'].dt.strftime('%Y-%m-%d')
            df_export_pred.to_excel(writer, index=False, sheet_name='Prévisions Échéances')
    return output.getvalue()

exp1, exp2 = st.columns(2)
with exp1:
    st.subheader("1. Rapport Complet Analytique")
    global_excel = generate_global_report(df_filtered, df_futur)
    st.download_button(label="📊 Télécharger le Rapport Global (Excel)", data=global_excel, file_name='rapport_tontine_complet.xlsx', mime='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet', use_container_width=True)

with exp2:
    st.subheader("2. Extraction brute CSV")
    csv_data = df_filtered.to_csv(index=False).encode('utf-8')
    st.download_button(label="📄 Télécharger les données filtrées (CSV)", data=csv_data, file_name='tontine_donnees_filtrees.csv', mime='text/csv', use_container_width=True)

In [ ]:
!curl ipv4.icanhazip.com

In [ ]:
import os
import time
import subprocess

print("🧹 1. Fermeture des anciennes instances pour libérer le port...")
!pkill -f streamlit > /dev/null 2>&1
!fuser -k 8501/tcp > /dev/null 2>&1
time.sleep(2)

print("🚀 2. Déploiement et exécution de l'application Streamlit...")
# Lancement de l'application avec les configurations nécessaires pour l'importateur Excel/CSV
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.enableCORS=false", "--server.enableXsrfProtection=false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)

print("🌐 3. Génération du lien d'accès public sécurisé...")
try:
    from google.colab import output
    # Demande à l'infrastructure interne de Google de créer un pont vers le port 8501
    public_url = output.eval_js("google.colab.kernel.proxyPort(8501)")

    print("\n" + "═"*60)
    print("🎉 APPLICATION DE TONTINE DÉPLOYÉE AVEC SUCCÈS !")
    print(f"👉 ACCÉDEZ AU DASHBOARD ICI : {public_url} 👈")
    print("═"*60 + "\n")
    print("📥 Note : Sur ce lien sécurisé, le module 'Importation des Données' acceptera vos fichiers sans aucune erreur.")
except ImportError:
    print("\n💡 Vous n'exécutez pas ce code depuis Google Colab.")
    print("L'application reste accessible localement à l'adresse suivante :")
    print("👉 http://localhost:8501 👈")